# Gradio 日！

今天我们将用简单到夸张的 Gradio 框架构建用户界面。

准备好享受乐趣吧！

请注意：你的 Gradio 界面可能显示为「深色模式」或「浅色模式」，取决于你的电脑设置。

In [ ]:
# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI

In [ ]:
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr # oh yeah!

In [ ]:
# 从名为 .env 的文件加载环境变量
# 打印密钥前缀以便调试
# 你可以选择任意提供商——或者全部用 Ollama

load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

In [ ]:
# 连接 OpenAI、Anthropic 和 Google；若不使用 Claude 或 Google，请注释掉对应行

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 用 OpenAI 兼容接口连接其它厂商：关键指定 base_url（服务地址）和 api_key
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# 把对 GPT-4.1-mini 的调用封装成一个简单函数

system_message = "You are a helpful assistant"

# 定义函数：把一组步骤打包，方便重复调用
def message_gpt(prompt):
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# 这可以揭示「训练截止日期」，即训练数据中最近的日期

message_gpt("What is today's date?")

## 用户界面时间！

In [ ]:
# 这里是一个简单函数

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
# launch：启动本地 Web 服务并打开演示页面
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">注意：使用 Gradio 的 Share 功能</h2>
            <span style="color:#900;">我马上要展示一种很酷的方式，把你的 Gradio UI 分享给别人。这会把你的 Gradio 应用作为演示部署到 Gradio 网站上，然后允许 Gradio 调用 'shout' 函数。它使用一种称为「HTTP 隧道」的高级技术（了解 ngrok 的人会很熟悉），许多杀毒软件和企业环境不允许使用。如果出错，跳过下一个单元格即可。<br/>
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 添加 share=True 意味着它可以公开访问
# 更持久的托管可用 HuggingFace 的 Spaces 平台，我们下周会涉及
# 注意：某些杀毒软件和企业防火墙可能不喜欢你使用 share=True。
# 如果你在工作环境或办公网络中，建议跳过此测试。

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# 添加 inbrowser=True 会自动打开新的浏览器窗口

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## 添加身份验证

Gradio 让用户名和密码变得非常容易

显然，如果你使用它，请从安全位置正确读取密码！至少使用你的 .env

In [ ]:
# launch：启动本地 Web 服务并打开演示页面
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("ed", "bananas"))

## 强制深色模式

Gradio 会根据浏览器和电脑设置显示浅色或深色模式。有办法强制 Gradio 使用深色模式，但 Gradio 不建议这样做，因为这应该由用户偏好决定（尤其出于无障碍考虑）。如果你仍想强制界面为深色模式，下面是做法。

In [ ]:
# 定义此变量，然后在创建 Interface 时传入 js=force_dark_mode

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
# launch：启动本地 Web 服务并打开演示页面
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# 再多加一点：

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

In [ ]:
# 现在——把函数从 "shout" 改成 "message_gpt"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

In [ ]:
# 让我们使用 Markdown
# 你是否在想：下面的代码并没有引用 system_message，设置它为何有用？
# 我利用了 system_message 是全局变量这一点，它会在 message_gpt 函数中被使用（去看看）
# 这不是很好的软件工程实践，但在 Jupyter Lab 研发中相当常见！

system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

In [ ]:
# 让我们创建一个流式返回结果的调用
# 如果你想复习生成器（"yield" 关键字），
# 请查看 guides 文件夹中的 Intermediate Python 指南

def stream_gpt(prompt):
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

In [ ]:
# 定义函数：把一组步骤打包，方便重复调用
def stream_claude(prompt):
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Claude 4.5 Sonnet", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_claude,
    title="Claude", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

## 现在来点高级的

如果你对生成器和 "yield" 还不确定，记得查看 Intermediate Python Guide

In [ ]:
# 定义函数：把一组步骤打包，方便重复调用
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Claude"]
        ], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

# 构建公司宣传册生成器

现在你知道怎么做了——很简单！

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">阅读接下来几个单元格之前</h2>
            <span style="color:#900;">
                先自己试一试——回到 week1 day5 的公司宣传册，在末尾加上 Gradio UI。然后再来看解决方案。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 导入本课程的 scraper 辅助函数：抓取网页正文或链接
from scraper import fetch_website_contents

In [ ]:

# 这又是典型的实验心态——我在修改上面用过的全局变量：

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
# 流式生成宣传册：边生成边刷新 Markdown 显示
def stream_brochure(company_name, url, model):
    yield ""
    # 拼出最终 prompt（提示词），送给模型
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    # 调用抓取函数，拉取网页正文或链接列表
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Claude"]
        ], 
    flagging_mode="never"
    )
# launch：启动本地 Web 服务并打开演示页面
view.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Gradio 资源</h2>
            <span style="color:#f71;">如果想深入了解 Gradio，请查看出色的文档——一个美妙的兔子洞。<br/>
            <a href="https://www.gradio.app/guides/quickstart">https://www.gradio.app/guides/quickstart</a><br/>Gradio 主要面向演示、原型和 MVP，我也经常用它为高级用户制作内部应用。
            </span>
        </td>
    </tr>
</table>